### Setup

In [0]:
print("gold_dq_started")

from pyspark.sql import functions as f
from datetime import datetime

In [0]:
S3_BUCKET = "s3://marketing-analytics-capstone"

S3_GOLD_DQ_METRICS_PATH = f"{S3_BUCKET}/monitoring/gold_data_quality_metrics"
S3_GOLD_QUARANTINE_PATH  = f"{S3_BUCKET}/gold/quarantine"

### Use Catalog

In [0]:
%sql
USE CATALOG marketing_analytics_capstone;
CREATE SCHEMA IF NOT EXISTS monitoring;

### Read Gold Tables

In [0]:
fact = spark.read.table("marketing_analytics_capstone.gold.fact_campaign_performance")
kpi_daily = spark.read.table("marketing_analytics_capstone.gold.kpi_daily_campaign_performance")
dim_campaign = spark.read.table("marketing_analytics_capstone.gold.dim_campaign")

## CORE METRICS

### Basic Volume Checks

In [0]:
metrics = fact.agg(
    f.count("*").alias("total_records"),
    f.sum("total_clicks").alias("total_clicks"),
    f.sum("total_impressions").alias("total_impressions")
).collect()[0]

total_records = metrics["total_records"]

print(f"Total Records: {total_records}")

## DATA QUALITY RULES

### Rule 1 – No Negative Metrics

In [0]:
dq_negative = fact.filter(
    (f.col("total_clicks") < 0) |
    (f.col("total_impressions") < 0) |
    (f.col("total_cost") < 0)
)

neg_count = dq_negative.count()

### Rule 2 – CTR Validity

In [0]:
dq_ctr = fact.filter(
    (f.col("avg_ctr") < 0) | (f.col("avg_ctr") > 1)
)

ctr_issues = dq_ctr.count()

### Rule 3 – Logical Check (Clicks ≤ Impressions)

In [0]:
dq_logic = fact.filter(
    f.col("total_clicks") > f.col("total_impressions")
)

logic_issues = dq_logic.count()

### Rule 4 – KPI Consistency Check
- Compare Silver vs Gold aggregation

In [0]:
silver = spark.read.table("marketing_analytics_capstone.silver.silver_campaigns") \
    .filter("dq_pass = true")

silver_agg = silver.groupBy("date").agg(
    f.sum("clicks").alias("clicks_silver")
)

gold_agg = kpi_daily.select(
    "date",
    f.col("total_clicks").alias("clicks_gold")
)

dq_consistency = silver_agg.join(gold_agg, "date") \
    .withColumn("diff", f.abs(f.col("clicks_silver") - f.col("clicks_gold"))) \
    .filter("diff > 0")

consistency_issues = dq_consistency.count()

### Rule 5 – Referential Integrity

In [0]:
dq_fk = fact.join(
    dim_campaign,
    "campaign_id",
    "left_anti"
)

fk_issues = dq_fk.count()

### Rule 6 – Missing Dates

In [0]:
dq_null_dates = fact.filter(f.col("date").isNull())

null_date_issues = dq_null_dates.count()

## AGGREGATE DQ METRICS

### Combine Metrics

In [0]:
dq_summary = {
    "negative_values": neg_count,
    "ctr_issues": ctr_issues,
    "logic_issues": logic_issues,
    "consistency_issues": consistency_issues,
    "fk_issues": fk_issues,
    "null_date_issues": null_date_issues
}

total_issues = sum(dq_summary.values())

print("DQ Summary:", dq_summary)
print("Total Issues:", total_issues)

In [0]:
dq_consistency.show(20, False)

## THRESHOLD LOGIC

### Threshold Config

In [0]:
MAX_TOTAL_ISSUES = 10
MAX_CRITICAL_ISSUES = 1   # e.g. FK or consistency issues

### Critical Issue Check

In [0]:
critical_issues = fk_issues + consistency_issues

### Decision Logic

In [0]:
if total_issues > MAX_TOTAL_ISSUES or critical_issues > MAX_CRITICAL_ISSUES:
    raise Exception(f"""
    GOLD DATA QUALITY FAILED

    Total Issues: {total_issues}
    Critical Issues: {critical_issues}

    Breakdown:
    {dq_summary}
    """)
else:
    print(f"""
    GOLD DATA QUALITY PASSED

    Total Issues: {total_issues}
    Critical Issues: {critical_issues}
    """)

## STORE METRICS (Monitoring)

### Persist DQ Metrics

In [0]:
from pyspark.sql import Row

metrics_row = [Row(
    run_timestamp=datetime.now(),
    layer="gold",
    total_records=total_records,
    total_issues=total_issues,
    critical_issues=critical_issues,
    negative_values=neg_count,
    ctr_issues=ctr_issues,
    logic_issues=logic_issues,
    consistency_issues=consistency_issues,
    fk_issues=fk_issues,
    null_date_issues=null_date_issues
)]

metrics_df = spark.createDataFrame(metrics_row) \
    .withColumn("run_date", f.current_date())

In [0]:
dq_failed_records = fact.filter(
    (f.col("total_clicks") < 0) |
    (f.col("total_impressions") < 0) |
    (f.col("total_cost") < 0) |
    (f.col("avg_ctr") < 0) | (f.col("avg_ctr") > 1) |
    (f.col("total_clicks") > f.col("total_impressions")) |
    (f.col("date").isNull())
)

In [0]:
dq_failed_records = dq_failed_records.withColumn(
    "dq_failure_reason",
    f.concat_ws(",",
        f.when(f.col("total_clicks") < 0, f.lit("negative_clicks")),
        f.when(f.col("total_impressions") < 0, f.lit("negative_impressions")),
        f.when(f.col("total_cost") < 0, f.lit("negative_cost")),
        f.when((f.col("avg_ctr") < 0) | (f.col("avg_ctr") > 1), f.lit("invalid_ctr")),
        f.when(f.col("total_clicks") > f.col("total_impressions"), f.lit("logic_error")),
        f.when(f.col("date").isNull(), f.lit("null_date"))
    )
).withColumn("run_date", f.current_date())

In [0]:
dq_failed_records.write \
    .format("delta") \
    .mode("append") \
    .option("path", S3_GOLD_QUARANTINE_PATH) \
    .saveAsTable("marketing_analytics_capstone.gold.quarantine_campaigns")

In [0]:
dq_failed_records.write \
    .mode("append") \
    .format("parquet") \
    .partitionBy("date") \
    .save(S3_GOLD_QUARANTINE_PATH + '_parquet')

In [0]:
metrics_df.write \
    .mode("append") \
    .saveAsTable("marketing_analytics_capstone.monitoring.gold_dq_metrics")

In [0]:
metrics_df.write \
    .mode("append") \
    .format("parquet") \
    .partitionBy("run_date") \
    .save(S3_GOLD_DQ_METRICS_PATH)